In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

In [ ]:
plt.figure(figsize=(10,6))
sns.set(style="whitegrid")

In [ ]:
df = pd.read_csv("demandas.csv")

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
df["data"] = pd.to_datetime(df["data"])

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
df.isna().sum().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.describe().T

In [ ]:
df.describe(include= "str").T

In [ ]:
df.head(2)

In [ ]:
df["ano"] = df["data"].dt.year
df["mes"] = df["data"].dt.month
df["dia"] = df["data"].dt.day
df["dia_semana"] = df["data"].dt.day_name()

In [ ]:
df

In [ ]:
df.columns

In [ ]:
df["preco_com_desconto"] = df["preco"] * (1 - df["desconto"] / 100)

In [ ]:
df.head()

In [ ]:
df["giro_de_estoque"] = df["unid_vendidas"] / df["nivel_estoque"]

In [ ]:
df["giro_de_estoque"].describe()

In [ ]:
df.groupby("categoria")["demanda"].agg(["mean","sum","std"]).sort_values(by = "sum", ascending=False)

In [ ]:
df.groupby(["regiao","sazonalidade"])["demanda"].mean()

In [ ]:
df.groupby("promocao")["demanda"].mean()

In [ ]:
pd.pivot_table(df, values = "demanda", index="mes", columns="categoria", aggfunc="mean")

In [ ]:
pivot = pd.pivot_table(df, values = "demanda", index="mes", columns="categoria", aggfunc="mean")

pivot_norm = pivot / pivot.max()

In [ ]:
sns.heatmap(data= pivot_norm, annot=True, cmap="YlGnBu")

In [ ]:
sns.histplot(df["demanda"], bins=20, kde=True)
plt.title("Distribuição da demanda")
plt.show()

In [ ]:
sns.scatterplot(data = df, x="nivel_estoque", y="unid_vendidas")
plt.title("Nivel de Estoque vs Unidades Vendidas")
plt.show()

In [ ]:
sns.boxplot(data=df, x="categoria", y="demanda")
plt.xticks(rotation = 45)
plt.title("Demanda por categoria")

In [ ]:
df.columns

In [ ]:
sns.boxplot(data=df, x="condicao_climatica", y="demanda")
plt.xticks(rotation = 45)
plt.title("Demanda por condição climatica")

In [ ]:
demanda_mensal = df.groupby("mes")["demanda"].mean()
demanda_mensal

In [ ]:
demanda_mensal.plot(kind= "bar")
plt.title("Media de demandas por mês")


In [ ]:
demanda_diaria = df.groupby("data")["demanda"].sum()
demanda_diaria

In [ ]:
demanda_diaria.plot()
plt.title("Total de demandas diária ao longo do tempo")
plt.xlabel("data")
plt.ylabel("demanda")

In [ ]:
sns.barplot(data= df, x="promocao", y="demanda")
plt.title("Impacto da promoção na demanda")
plt.show()

In [ ]:
sns.scatterplot(data= df, x="preco_com_desconto", y="demanda"),
plt.title("Preço com desconto vs Demanda")
plt.show()

In [ ]:
df.groupby("sazonalidade")["demanda"].mean().plot(kind= "bar", title="demanda por sazonalidade")
plt.show

In [ ]:
df.groupby("epidemia")["demanda"].mean().plot(kind= "bar", title= "impacto da epidemia sob demanda")
plt.show

In [ ]:
X = df['preco_com_desconto']
y = df['demanda']

# O statsmodels precisa que você adicione uma constante para o Beta 0
X = sm.add_constant(X)

modelo = sm.OLS(y, X).fit()
print(modelo.summary())

Entendendo o resultado: 
const ($\beta_0 = 108.57$): Se o preço fosse zero, a demanda média teórica seria de aproximadamente 108 unidades.
preco_com_desconto ($\beta_1 = -0.0689$): Este é o coeficiente de inclinação. Ele é negativo, o que faz total sentido: para cada 1 real de aumento no preço, a demanda cai cerca de 0,07 unidades.

P-valor (0.000): Como o valor é menor que 0.05, esses Betas são significativos. Há uma grande probabilidade que o preço realmente afeta a demanda, mesmo que o impacto total dele sozinho seja pequeno devido ao baixo R²

##### Dessa forma, temos: Demanda = 108.57 - 0.0689 x (preco_com_desconto)

In [ ]:
df.dtypes

In [ ]:
X = df[['promocao','nivel_estoque']]
y = df['demanda']

# O statsmodels precisa que você adicione uma constante para o Beta 0
X = sm.add_constant(X)

modelo = sm.OLS(y, X).fit()
print(modelo.summary())

In [ ]:
df.to_csv("preprocessamento_demanda.csv")